In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = (df["Churn"] == "Yes").astype(int)

# gender showed almost no effect in the EDA, so it is excluded from the features
X = df.drop(columns=["Churn", "gender"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

(5634, 18) (1409, 18)


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X_train.columns if c not in num_cols + ["SeniorCitizen"]]

# KNN and Logistic Regression: scale the numbers and one-hot encode the categories
prep_scaled = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
], remainder="passthrough")

# Random Forest: one-hot encode only (trees don't need scaling)
prep_tree = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
], remainder="passthrough")

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

models = {
    "KNN": Pipeline([("prep", prep_scaled), ("clf", KNeighborsClassifier())]),
    "Logistic Regression": Pipeline([("prep", prep_scaled), ("clf", LogisticRegression(max_iter=1000))]),
    "Random Forest": Pipeline([("prep", prep_tree), ("clf", RandomForestClassifier(random_state=42))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

rows = {}
for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows[name] = {m: scores[f"test_{m}"].mean() for m in scoring}

baseline = pd.DataFrame(rows).T.round(3)
baseline

,accuracy,precision,recall,f1,roc_auc
KNN,0.770,0.572,0.528,0.549,0.780
Logistic Regression,0.802,0.654,0.544,0.593,0.846
Random Forest,0.788,0.632,0.481,0.546,0.819


**Baseline results (5-fold stratified CV, default hyperparameters)**

- Logistic Regression achieved the best baseline performance across all metrics (F1 0.593, ROC-AUC 0.846), suggesting the churn signal is largely linear after scaling and one-hot encoding.
- Random Forest had the lowest recall (0.481), and KNN the lowest ROC-AUC (0.780).
- Recall is only about 0.48-0.54 for all models: each misses roughly half of the churners. Because churners are only 26.5% of the data, the models favour the majority class, so accuracy (0.77-0.80) overstates real performance. Class-imbalance handling and tuning against F1/recall will be explored next.

In [7]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV

smote_options = ["passthrough", SMOTE(random_state=42)]

searches = {
    "KNN": (
        ImbPipeline([("prep", prep_scaled), ("smote", "passthrough"),
                     ("clf", KNeighborsClassifier())]),
        {"smote": smote_options,
         "clf__n_neighbors": [5, 11, 21, 31, 41, 51],
         "clf__weights": ["uniform", "distance"],
         "clf__p": [1, 2]},
    ),
    "Logistic Regression": (
        ImbPipeline([("prep", prep_scaled), ("smote", "passthrough"),
                     ("clf", LogisticRegression(max_iter=2000))]),
        {"smote": smote_options,
         "clf__C": [0.001, 0.003, 0.01, 0.1, 1, 10],
         "clf__class_weight": [None, "balanced"]},
    ),
    "Random Forest": (
        ImbPipeline([("prep", prep_tree), ("smote", "passthrough"),
                     ("clf", RandomForestClassifier(random_state=42))]),
        {"smote": smote_options,
         "clf__n_estimators": [200, 400],
         "clf__max_depth": [4, 6, 8, 12, None],
         "clf__min_samples_leaf": [5, 10, 20, 40],
         "clf__class_weight": [None, "balanced"]},
    ),
}

In [8]:
best_models = {}
rows = {}
for name, (pipe, grid) in searches.items():
    gs = GridSearchCV(pipe, grid, scoring="f1", cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train)
    best_models[name] = gs.best_estimator_
    print(name, gs.best_params_, "\n")

    scores = cross_validate(gs.best_estimator_, X_train, y_train, cv=cv, scoring=scoring)
    rows[name] = {m: scores[f"test_{m}"].mean() for m in scoring}

tuned = pd.DataFrame(rows).T.round(3)
tuned

KNN {'clf__n_neighbors': 41, 'clf__p': 1, 'clf__weights': 'uniform', 'smote': SMOTE(random_state=42)} 

Logistic Regression {'clf__C': 0.01, 'clf__class_weight': None, 'smote': SMOTE(random_state=42)} 

Random Forest {'clf__class_weight': 'balanced', 'clf__max_depth': 8, 'clf__min_samples_leaf': 10, 'clf__n_estimators': 200, 'smote': 'passthrough'} 



,accuracy,precision,recall,f1,roc_auc
KNN,0.721,0.485,0.829,0.612,0.830
Logistic Regression,0.759,0.532,0.775,0.631,0.844
Random Forest,0.757,0.528,0.797,0.635,0.847


In [9]:
comparison = pd.concat({"Baseline": baseline, "Tuned": tuned}, axis=1)
comparison

Baseline                                    Tuned  \
                    accuracy precision recall     f1 roc_auc accuracy   
KNN                    0.770     0.572  0.528  0.549   0.780    0.721   
Logistic Regression    0.802     0.654  0.544  0.593   0.846    0.759   
Random Forest          0.788     0.632  0.481  0.546   0.819    0.757   

                                                     
                    precision recall     f1 roc_auc  
KNN                     0.485  0.829  0.612   0.830  
Logistic Regression     0.532  0.775  0.631   0.844  
Random Forest           0.528  0.797  0.635   0.847

**Tuned results (GridSearchCV, F1 scoring, 5-fold stratified CV)**

- Best settings: KNN k=41, Manhattan distance (p=1), uniform weights, with SMOTE; Logistic Regression C=0.01 with SMOTE; Random Forest 200 trees, max depth 8, minimum 10 samples per leaf, balanced class weights.
- Tuning improved F1 for all models: KNN 0.549 to 0.612, Logistic Regression 0.593 to 0.631, Random Forest 0.546 to 0.635.
- Recall rose substantially (KNN 0.829, Random Forest 0.797, Logistic Regression 0.775), at the cost of lower precision and accuracy. This trade-off is acceptable on the assumption that losing a customer costs more than an unnecessary retention offer.
- Random Forest and Logistic Regression are almost tied (F1 0.635 vs 0.631), so Random Forest cannot be considered clearly superior. Logistic Regression is also easier to interpret.
- All models selected some form of imbalance handling (SMOTE or class weighting).
- The search grid was widened after the first run found values at its boundaries; the selected values were unchanged.
- Limitation: these scores come from the folds used to select the parameters, so they may be slightly optimistic. The held-out test set provides the unbiased estimate.